# 26장 · 문의 성격을 읽고 다음 작업 고르기

회원은 규정을 물을 수도, 자신이 쓴 안내의 검토를 부탁할 수도 있습니다. “신청해 주세요”라는 요청에는 신청자만 아는 값이 필요합니다. 같은 작성 함수를 언제 부르고 언제 멈출지 결정해 봅니다.

## 자료로 해결할 빈칸과 사람에게 물을 빈칸

회원이 “오후 몇 시에 찾아가야 해요?”라고 물으면 규정에서 시간을 찾을 수 있습니다. 반면 “다음 주쯤 쓰고 싶은데 신청해 주세요”에서는 정확한 날이 사용자의 선택입니다. 화면에 “무엇을 더 알려 주세요?”를 매번 띄우면 회원은 규정에서 답할 수 있는 일까지 떠맡게 됩니다. 둘이 짝을 지어 문의에 밑줄을 긋고 원문 조항, 신청자, 안내문 작성자 중 누가 빈칸을 채울 수 있는지 먼저 적으세요. 분류 결과가 달라질 때 근거가 됩니다.

준비물은 실습 묶음의 `pre-mid/documents/sources/loan-rules.md`와 `pre-mid/documents/questions/loan-inquiries.md`입니다. 짝과 검토할 기록지는 `pre-mid/documents/workflow/workshop.md`에 있습니다.

## 분류 기준을 먼저 사람끼리 맞추기

“B형은 어디서 받아요?”에는 R2를 읽고 답할 수 있습니다. “제가 쓴 수령 안내가 맞나요?”에는 초안의 잘못된 문구를 짚어야 합니다. “이번 달에 하나 빌려 주세요”에는 기종과 특정 날짜가 빠져 있습니다. 세 경우를 각각 answer, review, clarify라는 작업 이름으로 정합니다. 두 가지 표현이 섞이면 초안 검토 요청인지, 신청 실행에 필요한 정보가 없는지 먼저 읽고 결정합니다.

## 실습 준비

8장에서 모델을 호출했던 Jupyter 커널을 선택하고, 이 장의 실습 묶음을 풀어 노트북과 pre-mid 폴더를 나란히 둡니다. 새로운 커널이라면 빈 셀에서 `%pip install -U openai`를 실행하고 수업용 API 키를 환경 변수로 설정한 뒤 시작하세요. 키는 노트북 셀이나 제출물에 적지 않습니다. 아래 함수에는 매번 지시와 이번에 읽을 자료를 전달합니다. 출력은 실행할 때마다 표현이 달라질 수 있습니다.

In [ ]:
from pathlib import Path
import os
from openai import OpenAI

# ZIP을 푼 폴더에서 노트북을 열면 이 경로에 네 가지 대여 조항이 있습니다.
rules = Path("pre-mid/documents/sources/loan-rules.md").read_text(encoding="utf-8")
model = os.environ.get("OPENAI_MODEL", "gpt-4.1-mini")
from getpass import getpass
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API 키: ")

client = OpenAI(timeout=45, max_retries=1)  # 8장에서 준비한 환경 변수 OPENAI_API_KEY를 읽습니다.

def ask(instruction, material):
    response = client.responses.create(
        model=model, instructions=instruction, input=material, store=False
    )
    return response.output_text

In [ ]:
incomplete = '대여 신청 좀 부탁드려요. 이번 달에 쓸 예정이에요.'
review_request = '신입 회원에게 올릴 이 안내가 규정에 맞는지 봐 주세요. ‘학생증을 친구에게 맡기면 대신 받을 수 있고, 기기는 이틀 뒤까지 가져오면 됩니다.’'
ordinary = "승인받은 회원은 어디에서 녹음기를 받나요?"

## 모델은 작업 이름을 제안하고 프로그램은 확인하기

분류 단계에서는 답안 작성까지 요청하지 않습니다. 다음 단계에 필요한 것은 허용한 이름 하나뿐입니다. 빈 응답이나 알 수 없는 이름으로 잘못된 작업을 실행하지 않도록 실제 받은 값을 확인합니다.

In [ ]:
route_instruction = """작업 이름 하나만 쓰세요.
answer: 규정을 읽어 답할 수 있는 질문.
review: 사용자가 작성한 안내문을 규정과 대조해 달라는 요청.
clarify: 신청에 필요한 기종 또는 정확한 날짜를 사용자가 주지 않음.
예약을 대신 처리하지 마세요."""
def select_route(message):
    chosen = ask(route_instruction, message).strip().lower()
    # 예기치 않은 이름은 기록하고 작성 단계로 넘기지 않습니다.
    if chosen not in {"answer", "review", "clarify"}:
        raise ValueError("작업 이름 확인 필요: " + chosen)
    return chosen

In [ ]:
route = select_route(incomplete)
print(route)

Q3에서 clarify가 나왔다면 실제 요청을 살펴보세요. “이번 달”은 기종도 특정 대여일도 알려 주지 않습니다. 반대로 “대여일은 언제까지 적나요?”는 날짜를 알려 달라는 신청이 아니라 규정 질문입니다. 이 차이를 분류 지시에 넣을지 새 입력으로 판단하세요.

## 선택된 함수만 실행하기

각 분기의 끝에서 바로 결과를 돌려줍니다. 추가 질문으로 끝난 뒤 공통 작성 코드를 또 실행하면 정보가 없는데도 답안이 하나 더 생깁니다. 아래 검토 경로는 잘못된 인용을 찾아 수정안을 쓰고, 일반 경로는 규정에 따라 답합니다.

In [ ]:
def handle(route, message):
    if route == "clarify":
        return ask("빠진 신청 정보만 짧게 물으세요. 신청이나 예약을 확정하지 마세요.", rules + "\n" + message)
    if route == "review":
        return ask("초안의 잘못된 문구를 인용하고 조항과 수정안을 보여 주세요.", rules + "\n" + message)
    if route == "answer":
        return ask("대여 규정에 따라 질문에 직접 답하고 조항을 적으세요.", rules + "\n" + message)
    raise ValueError("허용되지 않은 작업 이름")

In [ ]:
print(handle(route, incomplete))

올바른 추가 질문은 기종과 정확한 날짜를 묻습니다. “신청이 완료되었습니다”라는 말은 프로그램이 하지 않은 일을 약속하므로 문제가 됩니다. 판단과 실제로 수행한 행동을 나란히 보세요.

In [ ]:
chosen = select_route(review_request)
print(chosen)
print(handle(chosen, review_request))

Q2는 규정을 묻는 듯 보이지만 “이 안내가 맞는지” 검토해 달라고 했습니다. 결과가 review여도 수정할 두 구절이 모두 나왔는지 별도로 확인하세요. 작업 선택과 작업 결과의 정확성은 서로 다른 질문입니다.

## 실습 · 비슷하게 들리는 세 문장 분류하기

개인 실습 90분. (가) “연장 신청에 무엇이 필요하죠?” (나) “이번 달에 B형 신청해 주세요.” (다) “안내문에 ‘B형은 친구가 수령 가능’이라고 썼는데 그대로 올려도 돼요?”에 예상 경로를 먼저 붙이세요. 같은 뜻을 구어체로 바꿔 한 번 더 보냅니다. 다르게 분류된 한 쌍 또는 가장 헷갈린 두 문의의 차이를 원문에서 찾아 지시를 다듬고 재실행하세요.

In [ ]:
message = "이번 달에 B형 신청해 주세요."
# 기종은 알려 줬지만 날짜는 특정되지 않았습니다.

<details class="notebook-answer"><summary>모범 풀이와 대조 기준</summary>
<pre><code>for message in [&quot;연장 신청에 무엇이 필요하죠?&quot;, &quot;늘리려면 뭘 해야 하죠?&quot;,
                &quot;이번 달에 B형 신청해 주세요.&quot;, &quot;B형 좀 빌려줘요. 이달 중에요.&quot;,
                &quot;‘B형은 친구가 수령 가능’이라고 썼는데 그대로 올려도 돼요?&quot;,
                &quot;친구가 가져가도 된다고 공지하려는데 맞을까요?&quot;]:
    selected = select_route(message)
    print(message, selected)
    print(handle(selected, message))</code></pre>
<p>(가)는 answer, (나)는 clarify, (다)는 review가 적합합니다. (나)의 추가 질문은 날짜 하나면 됩니다. 기종까지 다시 물었다면 경로는 옳고 질문의 범위가 넓습니다. 경로명만 채점하면 이 문제를 놓칩니다.</p>
</details>

## 다음 작업에 가져갈 기록

예상 경로와 실제 경로가 달랐던 문장, 그 이유를 설명하는 표현을 보관하세요. 다음 장에서는 review 경로를 사실 검토와 표현 검토로 나눕니다.

[OpenAI Python SDK 사용 예](https://github.com/openai/openai-python)